# Two-tone coherence — full reference

Everything about this task in one place: the stimulus, the model it is testing, the geometry
that constrains it, the staircase, the design, the verification battery, the power, and a worked
run.

The short version is in `tcoh_two_tone_coherence.ipynb`. This one holds the arithmetic.

---

### Where it comes from

Elhilali, Ma, Micheyl, Oxenham & Shamma (2009, *Neuron* 61:317–329) argue that two sounds fuse
into one stream when their channels are temporally **coherent**, and split when they are not.
Two figures matter here:

- **Figure 8B** sweeps the onset asynchrony of two tones from synchrony to alternation and plots
  the model's segregation index λ₂/λ₁, which climbs from 0.01 to 0.93. A prediction, never tested
  behaviourally.
- **Figure 2** is their psychophysics: detect a temporal shift of the last tone in one of two
  sequences. Thresholds were 2–4 ms when the tones were synchronous and 10–20 ms when they were
  not — the same as with the low tone switched off entirely.

Figure 2 compares two states, and does so through a tempo difference rather than a fixed lag.
This task makes the lag a continuous variable and measures the threshold at each value.

### Scope

Two designs live in this package and they answer different questions.

- **`tcoh_pilot`** estimates a behavioural curve: threshold against onset lag. It has no
  controls. It does **not** establish temporal binding and cannot separate the mechanisms that
  predict such a curve.
- **`tcoh_core`** adds the controls that can. That is a three-sitting experiment.

Nothing here is listener data. Where a run is shown it is a simulated observer, labelled.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json, math
from pathlib import Path
REF='2a7f4c55aa0527cc9cb7b04afc2478a8d7a04ceb'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    import tcoh
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT=Path('/content')/('tcoh-'+REF[:12])
        if not ROOT.exists():
            subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
            subprocess.run(['git','-C',str(ROOT),'checkout','--detach',REF],check=True)
    else:
        ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tcoh/model.py').exists()),None)
    for _m in [k for k in list(sys.modules) if k=='tcoh' or k.startswith('tcoh.')]: del sys.modules[_m]
    sys.path.insert(0,str(ROOT))

get_ipython().run_line_magic('matplotlib','inline')
import numpy as np, matplotlib
matplotlib.rcParams.update({'figure.dpi':120,'font.size':9})
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown
from tcoh.config import (DEFAULT, Config, validate, conditions, interval_ms,
                         displaced_tone_clearance, max_safe_delta_ms, FUSION_MARGIN_MS)
from tcoh import model as M, stimulus as S, plots as P, verify as V, design as DS
CFG=DEFAULT; D=validate(CFG); CONDS=conditions(CFG)
# the repository root, taken from the package that was just imported rather than by searching
# upwards from the working directory. On Colab the clone lands in /content/tcoh-<ref>/ while the
# working directory stays /content, so the upward search finds nothing and raises StopIteration.
import tcoh as _tcoh
ROOT=Path(_tcoh.__file__).resolve().parent.parent
PILOT=Config.from_dict(json.loads((ROOT/'tcoh/configs/tcoh_pilot.json').read_text()))
def head(s): display(Markdown(s))
head(f"**core config** `{CFG.hash()}` &nbsp;·&nbsp; **pilot config** `{PILOT.hash()}`")

---
# 1. The stimulus

Two pure tones, low (A) and high (B), each an isochronous sequence. B's grid is the reference;
A's is the same grid displaced by a lag, expressed as **ΔT%** — a percentage of *half* the
period, so 0% is exact synchrony and 100% exact alternation.

In [ ]:
#@title every parameter that matters
rows=[("A, the low tone",f"{CFG.f_a_hz:.0f} Hz"),
      ("B, the high tone",f"{D.f_b_hz:.0f} Hz"),
      ("separation",f"{CFG.df_semitones:g} semitones = {CFG.df_semitones/12:.2f} octaves = {D.erbs_apart:.1f} ERB"),
      ("tone duration",f"{CFG.tone_ms:g} ms including {CFG.ramp_ms:g} ms raised-cosine ramps"),
      ("SOA within a channel",f"{CFG.soa_ms:g} ms  (duty cycle {CFG.duty:.2f})"),
      ("tones per channel",f"{CFG.n_precursor} precursors + 1 target = {CFG.n_tones}"),
      ("one interval",f"{interval_ms(CFG):.0f} ms"),
      ("gap between intervals",f"{CFG.isi_ms:g} ms"),
      ("level",f"{CFG.tone_level_db_spl:.0f} dB SPL per tone, roved ±{CFG.level_rove_db:g} dB per interval"),
      ("presentation","diotic" if not CFG.monaural else "monaural, left"),
      ("ΔT levels",", ".join(f"{p:g}%" for p in CFG.sweep_pcts)),
      ("lag in ms",", ".join(f"{CFG.lag_ms(p):.1f}" for p in CFG.sweep_pcts))]
head("| | |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a,b in rows))

In [ ]:
#@title one trial
P.trial(CFG); plt.show()

In [ ]:
#@title the exact onsets, in milliseconds
b=S.b_onsets(CFG)
head("B is the same grid in every condition. A is that grid plus the lag.\n")
lines=["| ΔT | lag | A onsets | B onsets |","|---|---|---|---|"]
for p in CFG.sweep_pcts:
    a=b+CFG.lag_ms(p)
    lines.append(f"| {p:g}% | {CFG.lag_ms(p):.2f} ms | {' '.join(f'{x:.0f}' for x in a)} | "
                 f"{' '.join(f'{x:.0f}' for x in b)} |")
head("\n".join(lines))

In [ ]:
#@title the sweep
P.schematic(CFG); plt.show()

## What it sounds like

Shift set well above threshold so it is audible. **The first interval is always the odd one out
in these clips.** In the experiment it is randomised.

In [ ]:
#@title one clip per ΔT
for c in [x for x in CONDS if x.a_kind=='coherent']:
    tr=S.build_trial(CFG,c,25.0,np.random.default_rng(4),target_position=1,direction=-1)
    head(f"**ΔT = {c.lag_pct:g}%** &nbsp; lag {CFG.lag_ms(c.lag_pct):.1f} ms")
    display(Audio(S.render_trial(CFG,tr,D),rate=CFG.sample_rate,normalize=False))

In [ ]:
#@title the same ΔT at four shift sizes
PCT=50.0
c=next(x for x in CONDS if x.a_kind=='coherent' and x.lag_pct==PCT)
for dl in (3.0,8.0,20.0,45.0):
    tr=S.build_trial(CFG,c,dl,np.random.default_rng(9),target_position=1,direction=-1)
    head(f"**δ = {dl:g} ms** at ΔT = {PCT:g}%")
    display(Audio(S.render_trial(CFG,tr,D),rate=CFG.sample_rate,normalize=False))

In [ ]:
#@title the controls and the ceiling, for comparison
for name,note in (('scr_50','scrambled: A present, its earlier tones at random times'),
                  ('b_only','no low tone at all — the ceiling')):
    c=next((x for x in CONDS if x.name==name),None)
    if c is None: continue
    tr=S.build_trial(CFG,c,25.0,np.random.default_rng(4),target_position=1,direction=-1)
    head(f"**{name}** — {note}")
    display(Audio(S.render_trial(CFG,tr,D),rate=CFG.sample_rate,normalize=False))

---
# 2. The invariants

Three properties make the comparison a test rather than a demonstration. They are checked below
by rendering waveforms and comparing samples, not by assertion.

In [ ]:
#@title checked, not asserted
d=D
rows=[]
ref=None; a_diff=0.0; b_diff=0.0; worst=0.0
for c in CONDS:
    tr=S.build_trial(CFG,c,8.0,np.random.default_rng(11),target_position=1,direction=-1)
    std,tgt=tr.second,tr.first
    worst=max(worst,abs((tgt.b_onsets_ms[-1]-std.b_onsets_ms[-1])+8.0))
    if c.a_kind!='absent':
        a1=V._render_channel(CFG,tr.first,d,'a',tr.phases); a2=V._render_channel(CFG,tr.second,d,'a',tr.phases)
        a_diff=max(a_diff,float(np.max(np.abs(a1-a2))))
    bs=V._render_channel(CFG,tr.second,d,'b',tr.phases)
    if ref is None: ref=bs
    elif bs.size==ref.size: b_diff=max(b_diff,float(np.max(np.abs(bs-ref))))
head(f"1. **Only the final B tone moves.** Largest deviation from the requested shift across "
     f"{len(CONDS)} conditions: `{worst:.2e}` ms.\n\n"
     f"2. **A is bit-identical between the two intervals**, so the low tone alone carries no "
     f"information about the answer. Max |difference|: `{a_diff:.3e}`.\n\n"
     f"3. **B is bit-identical across conditions** of the same length, so the high tone's own cue "
     f"cannot vary with ΔT. Max |difference|: `{b_diff:.3e}`.\n\n"
     "Together: the information in either channel *alone* is the same everywhere. Only the "
     "**relation** between them varies with ΔT, so no single-channel account can explain a ΔT "
     "effect.")

---
# 3. How large a shift the geometry allows

The first real session kept hitting the 45 ms ceiling, so the obvious move was to raise it.
Checking first turned up something worse than a low ceiling.

A **late** shift moves the displaced tone *towards* its own A partner, which lags it by exactly
the lag. At δ = lag the two coincide and the listener hears a chord the standard interval did
not contain. That is a different cue, not a bigger one — and it sits inside the measurable range
at small lags. An **early** shift moves the tone away from its partner instead.

In [ ]:
#@title how close the displaced tone comes to an A tone
rows=["| ΔT | direction | "+" | ".join(f"δ={d_}" for d_ in (10,20,30,45,60,70))+" |",
      "|---|---|"+"---|"*6]
for p in (0.,25.,50.,75.,100.):
    for sgn,nm in ((-1,'early'),(+1,'late')):
        cells=[]
        for d_ in (10,20,30,45,60,70):
            g=displaced_tone_clearance(CFG,p,float(d_),sgn)
            mark="**" if (p>0 and (g['makes_more_synchronous'] or g['a_sep_ms']<FUSION_MARGIN_MS)) else ""
            cells.append(f"{mark}{g['a_sep_ms']:.1f}{mark}")
        rows.append(f"| {p:g}% | {nm} | "+" | ".join(cells)+" |")
head("Separation in ms from the nearest A tone. **Bold** = closer than the "
     f"{FUSION_MARGIN_MS:.0f} ms fusion margin, or more synchronous than the standard was.\n\n"
     +"\n".join(rows))

In [ ]:
#@title the limit, per direction
pcts=[0.,25.,50.,75.,100.]
rows=["| shift direction | largest usable δ |","|---|---|"]
for sgn,nm in ((-1,'early only'),(+1,'late only')):
    rows.append(f"| {nm} | **{max_safe_delta_ms(CFG,pcts,sgn):.0f} ms** |")
rows.append(f"| collision with the preceding B tone | {CFG.soa_ms-CFG.tone_ms:.0f} ms |")
head("\n".join(rows))
head(f"So the pilot shifts **early only** and stops at **{PILOT.delta_max_ms:g} ms**. "
     "Fifty is barely more than forty-five: past that the repetition period itself is the "
     "limit, and changing it is a different experiment.")

In [ ]:
#@title nothing clips
pk=0.
for c in CONDS:
    for dl in (CFG.delta_min_ms,20.,PILOT.delta_max_ms):
        tr=S.build_trial(PILOT,c,dl,np.random.default_rng(1)) if c.a_kind=='coherent' else None
        if tr is None: continue
        iv=S.Interval(tr.first.a_onsets_ms,tr.first.b_onsets_ms,tr.first.delta_ms,
                      PILOT.level_rove_db,c.a_kind,c.lag_pct)
        pk=max(pk,float(np.max(np.abs(S.render_interval(PILOT,iv,validate(PILOT),tr.phases)))))
head(f"Peak with the level rove at its maximum: **{pk:.3f} FS**, "
     f"{20*np.log10(1/pk):.1f} dB of headroom.")

---
# 4. The model

Re-implemented for two channels, so the prediction is for *these* stimuli rather than read off
the published figure with a ruler.

The pipeline: each channel's envelope through a bank of rate filters (2–32 Hz, six phases),
cross-correlated and summed over the bank into a 2×2 coherence matrix, then decomposed. With two
equal-power channels the answer is closed form: λ₂/λ₁ = (1−r)/(1+r), where r is the normalised
cross-channel correlation.

In [ ]:
#@title the filter bank
fs=1000.
fig,axes=plt.subplots(1,2,figsize=(10,3))
for w in M.RATES_HZ:
    h=M.rate_filter(w,0.0,fs); axes[0].plot(np.arange(h.size)/fs*1000,h/np.abs(h).max(),lw=1.3,label=f'{w:g} Hz')
axes[0].set_xlim(0,600); axes[0].set_xlabel('ms'); axes[0].set_ylabel('normalised')
axes[0].legend(fontsize=7,frameon=False); axes[0].set_title('impulse responses, phase 0',fontsize=9)
for w in M.RATES_HZ:
    h=M.rate_filter(w,0.0,fs); f=np.fft.rfftfreq(4096,1/fs); mg=np.abs(np.fft.rfft(h,4096))
    axes[1].semilogx(f[1:],mg[1:]/mg.max(),lw=1.3)
axes[1].set_xlim(0.5,80); axes[1].set_xlabel('modulation rate (Hz)')
axes[1].set_title('bandpass, so co-activation alone counts for nothing',fontsize=9)
for ax in axes:
    for s_ in ('top','right'): ax.spines[s_].set_visible(False)
plt.tight_layout(); plt.show()

### The one place the published method had to be interpreted

Taken as bare arithmetic, the equation does not reproduce the paper's own numbers. Two perfectly
alternating channels are *anti*-correlated, not uncorrelated, and the signed product gives 0.16
where the paper reports 0.93, with a non-monotone curve.

The paper says what it means instead: the off-diagonals "are at zero for the alternating
sequence", the operation is called **coincidence**, and it is done by **coincidence detectors**.
A coincidence detector fires when both inputs are active at once and is otherwise silent — a
rectified product. That reading reproduces both stated values.

In [ ]:
#@title all three readings against the published values
r=M.reproduce_figure8()
rows=["| reading | ΔT=100% | ΔT=0% | monotone |","|---|---|---|---|"]
for k,v in r['readings'].items():
    rows.append(f"| {'**'+k+'**' if k==r['best'] else k} | {v['alternating']:.3f} | "
                f"{v['synchronous']:.4f} | {v['monotone']} |")
rows.append(f"| _published_ | _{M.PUBLISHED['alternating']}_ | _{M.PUBLISHED['synchronous']}_ | _yes_ |")
head("\n".join(rows)+f"\n\nThis package uses **{r['best']}**, flagged as an inference wherever it matters.")

In [ ]:
#@title what the model is handed, and what it returns
P.envelopes(CFG); plt.show()

### The duty cycle is not a free choice

ΔT is only an ordered axis when the tone fills half the period. Put a gap inside each channel and
the predicted index **peaks near 75% and comes back down**, so a monotone behavioural result
would confirm nothing. The validator refuses other duty cycles.

In [ ]:
#@title monotone only at 0.5
scan=M.duty_cycle_scan(soa_ms=CFG.soa_ms,n_tones=CFG.n_tones)
rows=["| tone / SOA | monotone in ΔT? | index peaks at |","|---|---|---|"]
for duty,v in scan.items():
    mk='**'+format(duty,'.3f')+'**' if abs(duty-CFG.duty)<1e-9 else format(duty,'.3f')
    rows.append(f"| {mk} | {'yes' if v['monotone'] else '**no**'} | ΔT = {v['argmax_pct']:.0f}% |")
head("\n".join(rows))
fig,ax=plt.subplots(figsize=(6,3.6))
for duty,v in scan.items():
    ax.plot([0,12.5,25,37.5,50,62.5,75,87.5,100],v['ratios'],'o-',ms=3,lw=1.6,
            label=f"{duty:.3f}"+(" (used)" if abs(duty-CFG.duty)<1e-9 else ""))
ax.set_xlabel('ΔT (%)'); ax.set_ylabel('predicted λ₂/λ₁'); ax.legend(fontsize=7,frameon=False,title='tone/SOA')
for s_ in ('top','right'): ax.spines[s_].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
#@title the prediction, and how much of it is robust
P.prediction(CFG); plt.show()
band=M.prediction_band(sorted(CFG.sweep_pcts),tone_ms=CFG.tone_ms,soa_ms=CFG.soa_ms,n_tones=CFG.n_tones)
rows=["| ΔT | index | range across 8 readings |","|---|---|---|"]
for i,p in enumerate(sorted(CFG.sweep_pcts)):
    rows.append(f"| {p:g}% | {band['mid'][i]:.3f} | {band['lo'][i]:.3f} – {band['hi'][i]:.3f} |")
head("\n".join(rows)+f"\n\nAll monotone: **{band['all_monotone']}**. All agreeing on the ordering "
     f"of the ΔT levels: **{band['all_agree_on_order']}**. The ORDER is predicted robustly. "
     "The heights are not, which is why the shape is never treated as evidence.")

In [ ]:
#@title the index for every condition as actually built
rows=["| condition | ΔT | A sequence | model λ₂/λ₁ |","|---|---|---|---|"]
for c in CONDS:
    v=D.model_by_condition[c.name]
    rows.append(f"| `{c.name}` | {c.lag_pct:g}% | {c.a_kind} | "+("—" if v!=v else f"{v:.3f}")+" |")
head("\n".join(rows))

---
# 5. Why the controls decide it, and why a rising curve does not

Three accounts all predict thresholds rising with ΔT:

1. **temporal coherence** — the tones stop being one object, so the low tone stops being usable;
2. **interval discrimination** — the A–B interval the listener judges grows with ΔT, and judging
   a change in a longer interval is harder, with no streaming involved;
3. **local acoustic overlap** — the two tones simply overlap less.

So "threshold rises with ΔT" settles nothing. The controls hold the **final A–B interval exactly**
and remove only the low tone's *sequence*. Accounts 2 and 3 depend on nothing but that final
pair, so both predict a **flat zero difference**. Coherence predicts a large difference at
synchrony that vanishes at alternation.

In [ ]:
#@title the predicted interaction
coh={c.lag_pct:D.model_by_condition[c.name] for c in CONDS if c.a_kind=='coherent'}
for kind in ('scrambled','pair_only'):
    ctl={c.lag_pct:D.model_by_condition[c.name] for c in CONDS if c.a_kind==kind}
    sh=sorted(set(coh)&set(ctl))
    if len(sh)<2: continue
    rows=[f"**{kind}**\n","| ΔT | coherent | control | difference |","|---|---|---|---|"]
    for p in sh: rows.append(f"| {p:g}% | {coh[p]:.3f} | {ctl[p]:.3f} | **{ctl[p]-coh[p]:+.2f}** |")
    head("\n".join(rows))
head("Positive at synchrony, falling to zero or below at alternation. "
     "A pedestal or overlap account predicts a column of zeros.")

---
# 6. The staircase

Elhilali et al.'s rule, implemented literally: 3-down 1-up on a multiplicative step, ×4 until the
first reversal, ×2 for two more, then ×√2, stopping at the sixth reversal at ×√2. Threshold is
the geometric mean of those six.

One discrepancy in their Methods is followed rather than silently resolved: the same paragraph
calls the rule "three-down one-up … 79.4%" and describes stepping down after **two** consecutive
correct responses, which converges on 70.7%. We follow the stated target.

In [ ]:
#@title does the rule recover a threshold it was given?
from tcoh.track import simulate
r=simulate(CFG,thresholds_ms=(2.,4.,8.,16.,24.),sigmas=(0.35,0.6,0.9),n_runs=150,seed=5)
rows=["| true θ | slope | recovered | bias | spread sd(log) | median trials |","|---|---|---|---|---|---|"]
for v in r['cells'].values():
    rows.append(f"| {v['true_ms']:.0f} ms | {v['sigma']:.2f} | {v['recovered_geomean_ms']:.2f} ms | "
                f"{v['bias_pct']:+.0f}% | {v['sd_log']:.2f} | {v['median_trials']:.0f} |")
head("\n".join(rows))
head(f"Worst bias **{r['worst_abs_bias_pct']:+.0f}%**, and it is a slight UNDERestimate at every "
     "threshold, so it largely cancels in the log ratios the analysis is built from. "
     f"Non-convergence {r['max_fail_rate']:.0%}.")

In [ ]:
#@title a reversal on the clamp is not a reversal
head(f"A track that runs up against `delta_max_ms` cannot take the step the rule calls for, so the "
     f"value recorded is the clamp. Averaging those into the threshold produces a number that is "
     f"partly a statement about the ceiling. The analysis marks such conditions **CENSORED**, "
     f"reports them as bounds, and keeps them out of κ and the tests.\n\n"
     "This is not hypothetical: in the first recorded session, two of the six reversals averaged "
     "into two conditions were exactly 45.0 ms.")

---
# 7. The order of the session

Three things have to be true of the order: no condition confounded with time, no condition
identifiable trial to trial, and some way of knowing the listener was awake.

In [ ]:
#@title rounds, blocks and serial position
for cfg_,nm in ((PILOT,'tcoh_pilot'),(CFG,'tcoh_core')):
    des=DS.make_design(cfg_,'P01',1); a=DS.audit_design(cfg_,des); e=DS.duration_estimate(cfg_)
    head(f"**{nm}** — {a['n_tracks']} tracks in {des['n_blocks']} blocks of "
         f"{des['tracks_per_block']}, {cfg_.tracks_per_condition} round(s)\n\n"
         f"- every condition contributes exactly one track per round (mean round index varies by "
         f"{a['round_balance_spread']:.0e}), so each gets one track in each "
         f"{1/cfg_.tracks_per_condition:.0%} of the session\n"
         f"- within-round serial-position spread **{a['serial_position_spread']:.3f}** of the session\n"
         f"- longest planned run of one condition **{a['longest_condition_run']}** "
         f"(limit {a['max_same_condition_run']})\n"
         f"- catch trials planned at {a['catch_rate_planned']:.1%}")

### Catch trials, and why the first design of them was wrong

A catch trial is a supra-threshold probe that anyone attending should get. The first version used
a fixed 45 ms shift *in whichever condition hosted it* — which is eight times threshold in an
easy condition and barely above it in a hard one. Its difficulty tracked the condition, so a miss
meant nothing about attention. Both misses in the first real session were in the two hardest
conditions.

`catch_at_pct` fixes it: every probe is built from one easy condition wherever it lands. The
listener cannot pick them out, because trials at that lag occur normally anyway.

In [ ]:
#@title what each preset does about it
rows=["| preset | catch rate | probe δ | probe built from |","|---|---|---|---|"]
for nm in ('tcoh_pilot','tcoh_core'):
    c_=Config.from_dict(json.loads((ROOT/f'tcoh/configs/{nm}.json').read_text()))
    src=('the hosting condition — **not a valid lapse measure**' if c_.catch_at_pct is None
         else f'ΔT = {c_.catch_at_pct:g}% only')
    rows.append(f"| `{nm}` | {c_.catch_rate:.0%} | {c_.catch_delta_ms:g} ms | {src} |")
head("\n".join(rows))

---
# 8. Everything checkable without a listener

In [ ]:
#@title the battery
for section,checks in V.run_battery(CFG,quick=True).items():
    head(f"**{section}**")
    for c in checks:
        head(f"- {'✅' if c.passed else '❌'} {c.name}  \n  <small>{c.detail}</small>")

---
# 9. Power

The whole pipeline run against simulated listeners generated by the hypothesis and by each rival.
**Read the middle row**: the Weber rival produces the "threshold rises with ΔT" result on every
single simulated session. That is the reason H2 exists and the reason H1 is never reported as
though it settled something.

| generating truth | H1 fires | H2 fires |
|---|---|---|
| coherence (the hypothesis) | 100% | **87%** |
| pedestal (the Weber rival) | **100%** | 4% |
| null (nothing depends on ΔT) | 6% | 3% |

Cutting the matched control from five ΔT levels to three keeps H1 at 99% and drops H2 to **9%**.
That version is a screen, not a test, and the validator says so on every run.

The cell below re-measures this. It takes a few minutes; the numbers above came from 100
simulated sessions per truth.

In [ ]:
#@title re-measure it (slow — set SESSIONS higher for a tighter estimate)
SESSIONS=20
from tcoh.analysis import power
r=power(CFG,n_sessions=SESSIONS,n_boot=300,seed=31)
rows=["| generating truth | H1 fires | H2 fires | what it should be |","|---|---|---|---|"]
want={'coherence':'both high','pedestal':'H1 high, H2 near 5%','null':'both near 5%'}
for m,v in r['modes'].items():
    rows.append(f"| {m} | {v['H1_rate']:.0%} | {v['H2_rate']:.0%} | {want.get(m,'')} |")
head("\n".join(rows)+f"\n\n<small>{SESSIONS} sessions per truth — noisy at this size.</small>")

---
# 10. The descriptive pilot

Five coherent conditions, two tracks each, ten tracks, no controls. Its single output is a
threshold curve. The axis runs 100% alternation on the left to 0% synchrony on the right so it
can be laid beside Figure 8B — **that is the only relationship claimed between them.** What is
plotted is a detection threshold in milliseconds. It is not λ₂/λ₁ and it is not a rescaling of
one, and no B-only normalisation is computed, because this preset does not measure a ceiling.

In [ ]:
#@title the pilot design and what it costs
d_=validate(PILOT); e=DS.duration_estimate(PILOT)
head(f"**{len(d_.conditions)} conditions × {PILOT.tracks_per_condition} tracks = {d_.n_tracks} tracks**, "
     f"shifts {PILOT.delta_direction} only, δ capped at {PILOT.delta_max_ms:g} ms\n\n"
     f"| | |\n|---|---|\n"
     f"| estimate | **{e['total_minutes']:.0f} min** — {e['setup_minutes']:.0f} setup + "
     f"{e['practice_minutes']:.0f} practice + {e['main_minutes']:.0f} trials + {e['break_minutes']:.0f} break |\n"
     f"| expected trials | {e['n_trials']} |\n"
     f"| worst case | {e['worst_case_minutes']:.0f} min / {e['max_trials']} trials if every track ran to its cap |\n")
head("```bash\npython -m tcoh run --config tcoh/configs/tcoh_pilot.json --data data --code P01\n```")

In [ ]:
#@title a simulated pilot run, end to end
import tempfile
from tcoh.runner import Runner
from tcoh.analysis import load, thresholds
r=Runner(PILOT,Path(tempfile.mkdtemp()),audio=False,auto='coherence',seed=11)
sdir=r.run(code='SIM')
res=thresholds(load([sdir])[0],PILOT)
P.pilot_curve(PILOT,res,simulated=True); plt.show()

In [ ]:
#@title the same numbers as a table
rows=["| ΔT | track 1 | track 2 | geometric mean |","|---|---|---|---|"]
for p in sorted(PILOT.sweep_pcts):
    v=next(x for x in res.values() if x.a_kind=='coherent' and abs(x.lag_pct-p)<1e-9)
    t=[f"{x:.2f}" for x in v.track_thresholds] + ["—"]*(2-len(v.track_thresholds))
    gm=f"**{v.geomean_ms:.2f} ms**" if v.geomean_ms else (f"≥{v.bound_ms:.1f} ms (censored)" if v.censored else "—")
    rows.append(f"| {p:g}% | {t[0]} | {t[1]} | {gm} |")
head("\n".join(rows)+"\n\n<small>SIMULATED observer.</small>")

---
# 11. A full run of the controlled design, and the same under the rival

Both simulated. The point is that the analysis reaches different verdicts.

In [ ]:
#@title the full report under the hypothesis
from tcoh.analysis import analyse, interaction_test
r2=Runner(CFG,Path(tempfile.mkdtemp()),audio=False,auto='coherence',seed=11)
s2=r2.run(code='SIM')
print(analyse([s2],n_boot=1000))

In [ ]:
#@title and under the Weber rival
r3=Runner(CFG,Path(tempfile.mkdtemp()),audio=False,auto='pedestal',seed=11)
s3=r3.run(code='SIM')
i2=interaction_test(thresholds(load([s3])[0],CFG),CFG,n_boot=1000)
i1=interaction_test(thresholds(load([s2])[0],CFG),CFG,n_boot=1000)
rows=["| generating truth | H2 slope | permutation p | verdict |","|---|---|---|---|"]
for nm,t in (('coherence',i1),('pedestal rival',i2)):
    rows.append(f"| {nm} | {t['slope_per_pct']*100:+.2f} | {t['p_slope_negative']:.4f} | "
                f"{'**consistent with coherence**' if t['p_slope_negative']<0.05 else 'not resolved'} |")
head("\n".join(rows))

In [ ]:
#@title your own session, if you have one
SESSION=''   # e.g. 'data/P01/tcoh_session_02'
p_=Path(SESSION) if SESSION else None
if p_ and (p_/'trials.csv').exists():
    print(analyse([p_],n_boot=2000))
else:
    head("_No session path set. Put one in `SESSION` above to analyse real data. "
         "Recorded sessions live under `data/`, which is never committed._")

---
# 12. What this does not establish

- **The shape test is close to worthless.** Over these ΔT levels the model curve and a straight
  line correlate at 0.988 and differ by at most 0.15 once both are rescaled. No realistic amount
  of data separates them, and the report says so where it prints the comparison.
- **Onset lag and acoustic overlap are perfectly confounded** at a 50% duty cycle: ΔT% is exactly
  100 × (1 − overlap fraction). Separating them needs a duty-cycle manipulation — a different
  experiment, which the config already supports via `allow_nonmonotone_duty`.
- **The controls are imperfect in different directions.** The scrambled one matches tone count and
  energy but its own coherence is not flat across ΔT; the pair-only one has no A sequence at all
  but holds five fewer tones. The argument needs both to agree.
- **κ is normalised by two conditions from the same session**, so a bad floor or ceiling moves the
  whole curve. Both are printed in milliseconds beside it.
- **The pilot has no controls at all** and is descriptive by design.
- **Shifts are early only in the pilot**, so it measures the early-shift threshold. Whether the
  late-shift threshold differs is not answered here, and cannot be asked cleanly at intermediate
  ΔT for the reason in section 3.
- **One listener generalises to one listener.**

The pre-registration — hypotheses, decision rules, exclusion criteria, power — is in
`tcoh/PREREGISTRATION.md`, written before any listener was run.

---

# 13. Running it

```bash
python -m tcoh design  --config tcoh/configs/tcoh_pilot.json   # conditions, geometry, duration
python -m tcoh verify  --config tcoh/configs/tcoh_pilot.json   # 21 checks, no listener needed
python -m tcoh run     --config tcoh/configs/tcoh_pilot.json --data data --code P01
python -m tcoh run     --config tcoh/configs/tcoh_pilot.json --data data --code P01 --resume
python -m tcoh plots   data/P01/tcoh_session_02 --config tcoh/configs/tcoh_pilot.json --out <dir>
```

Sessions resume: press `q` to stop, `--resume` to continue. Every condition contributes one track
per round, so a part-finished session is still balanced.